# Classificação de Imagens por CNN

Este notebook foi organizado em seções separadas por arquitetura de CNN para facilitar a execução no Google Colab.

Modelos incluídos:

- DenseNet121: baseline do artigo original
- ResNet50: CNN clássica e muito citada
- ConvNeXt V2: CNN moderna


A ideia é manter a base comum no início e deixar cada modelo em um bloco próprio.

In [2]:
# ============================================================
# Base comum para todos os experimentos
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================

import os
import numpy as np
import pandas as pd
import kagglehub
from PIL import Image
from torchvision import transforms as T

from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# ============================================================
# 2. DOWNLOAD E LEITURA DO DATASET
# ============================================================

path = kagglehub.dataset_download("mahyeks/almond-varieties")
dataset_dir = os.path.join(path, "dataset")
valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

classes = sorted(
    nome for nome in os.listdir(dataset_dir)
    if os.path.isdir(os.path.join(dataset_dir, nome))
)

image_paths, labels = [], []
for classe in classes:
    classe_dir = os.path.join(dataset_dir, classe)
    for nome_arquivo in sorted(os.listdir(classe_dir)):
        if nome_arquivo.lower().endswith(valid_extensions):
            image_paths.append(os.path.join(classe_dir, nome_arquivo))
            labels.append(classe)

base_df = pd.DataFrame({"image_path": image_paths, "label": labels})
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(base_df["label"])

print("Base carregada com sucesso.")
print(base_df["label"].value_counts())

# ============================================================
# 3. DIVISÃO TREINO / TESTE  (originais apenas)
# ============================================================

X_train_paths, X_test_paths, y_train, y_test = train_test_split(
    base_df["image_path"].values,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)

print(f"\nDivisão treino/teste:")
print(f"  Treino (originais) : {len(X_train_paths)} imagens")
print(f"  Teste  (originais) : {len(X_test_paths)} imagens")

# ============================================================
# 4. DATA AUGMENTATION  (apenas no conjunto de treino)
# ============================================================

augmentation_ops = {
    "hflip":  T.RandomHorizontalFlip(p=1.0),
    "vflip":  T.RandomVerticalFlip(p=1.0),
    "rot30":  T.RandomRotation(degrees=30),
    "jitter": T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    "blur":   T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    "affine": T.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.85, 1.15)),
}

aug_dir = "augmented_train"
os.makedirs(aug_dir, exist_ok=True)

augmented_paths  = []
augmented_labels = []
total_originais  = len(X_train_paths)

print(f"\nAplicando data augmentation em {total_originais} imagens de treino...")
print(f"Operações: {list(augmentation_ops.keys())}\n")

for idx, (img_path, label) in enumerate(zip(X_train_paths, y_train)):
    img       = Image.open(img_path).convert("RGB")
    base_name = os.path.splitext(os.path.basename(img_path))[0]

    for op_name, transform in augmentation_ops.items():
        aug_img   = transform(img)
        save_path = os.path.join(aug_dir, f"{base_name}_{op_name}.jpg")
        aug_img.save(save_path, quality=95)
        augmented_paths.append(save_path)
        augmented_labels.append(label)

    if (idx + 1) % 100 == 0 or (idx + 1) == total_originais:
        print(f"  ✓ {idx + 1}/{total_originais} imagens processadas")

# Concatena originais + aumentadas → conjunto de treino completo
X_train_paths = np.concatenate([X_train_paths, augmented_paths])
y_train       = np.concatenate([y_train, augmented_labels])

# ============================================================
# 5. HELPERS DE GRUPO  (usados no split abaixo e nos modelos)
# ============================================================

SUFIXOS_AUG = ["_hflip", "_vflip", "_rot30", "_jitter", "_blur", "_affine"]

def extrair_nome_base(path: str) -> str:
    """Nome do arquivo sem extensão e sem sufixo de augmentation."""
    nome = os.path.splitext(os.path.basename(path))[0]
    for suf in SUFIXOS_AUG:
        if nome.endswith(suf):
            return nome[:-len(suf)]
    return nome

def is_original(path: str) -> bool:
    """True se a imagem não tiver sufixo de augmentation."""
    nome = os.path.splitext(os.path.basename(path))[0]
    return not any(nome.endswith(s) for s in SUFIXOS_AUG)

# ============================================================
# 6. DIVISÃO TREINO / VALIDAÇÃO  (compartilhada entre modelos)
#
# StratifiedGroupKFold garante que original + todas as suas
# versões aumentadas ficam SEMPRE no mesmo split, eliminando
# o vazamento de dados que ocorria com train_test_split simples.
#
# A validação é filtrada para apenas imagens originais,
# alinhando-a com o conjunto de teste (100% originais) e
# tornando a val_accuracy uma estimativa honesta.
#
# Variáveis exportadas para as células de cada modelo:
#   X_train_split / y_train_split  → treino  (originais + aumentadas)
#   X_val_paths   / y_val          → validação (apenas originais)
#   X_test_paths  / y_test         → teste     (apenas originais)
# ============================================================

groups  = np.array([extrair_nome_base(p) for p in X_train_paths])
sgkf    = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
dummy_X = np.arange(len(X_train_paths)).reshape(-1, 1)

tr_idx, val_idx = next(sgkf.split(dummy_X, y_train, groups))

X_train_split = X_train_paths[tr_idx]
y_train_split = y_train[tr_idx]

# Mantém no val apenas os originais cujo grupo caiu neste fold
orig_mask    = np.array([is_original(X_train_paths[i]) for i in val_idx])
val_idx_orig = val_idx[orig_mask]

X_val_paths = X_train_paths[val_idx_orig]
y_val       = y_train[val_idx_orig]

# ============================================================
# 7. RESUMO FINAL
# ============================================================

print(f"\n{'=' * 55}")
print("RESUMO DO DATASET")
print(f"{'=' * 55}")
print(f"Imagens originais de treino  : {total_originais}")
print(f"Imagens geradas (aug)        : {len(augmented_paths)}")
print(f"Total treino  (X_train_paths): {len(X_train_paths)}")
print(f"─── split para modelos ──────────────────────────────")
print(f"  X_train_split              : {len(X_train_split)} "
      f"(originais + aumentadas)")
print(f"  X_val_paths                : {len(X_val_paths)} "
      f"(apenas originais)")
print(f"  X_test_paths               : {len(X_test_paths)} "
      f"(apenas originais)")

print(f"\nDistribuição por classe (treino + aug):")
unique, counts = np.unique(y_train, return_counts=True)
for cls_idx, count in zip(unique, counts):
    print(f"  {label_encoder.classes_[cls_idx]:<20}: {count}")

Using Colab cache for faster access to the 'almond-varieties' dataset.
Base carregada com sucesso.
label
KAPADOKYA    465
AK           401
SIRA         384
NURLU        306
Name: count, dtype: int64

Divisão treino/teste:
  Treino (originais) : 1089 imagens
  Teste  (originais) : 467 imagens

Aplicando data augmentation em 1089 imagens de treino...
Operações: ['hflip', 'vflip', 'rot30', 'jitter', 'blur', 'affine']

  ✓ 100/1089 imagens processadas
  ✓ 200/1089 imagens processadas
  ✓ 300/1089 imagens processadas
  ✓ 400/1089 imagens processadas
  ✓ 500/1089 imagens processadas
  ✓ 600/1089 imagens processadas
  ✓ 700/1089 imagens processadas
  ✓ 800/1089 imagens processadas
  ✓ 900/1089 imagens processadas
  ✓ 1000/1089 imagens processadas
  ✓ 1089/1089 imagens processadas

RESUMO DO DATASET
Imagens originais de treino  : 1089
Imagens geradas (aug)        : 6534
Total treino  (X_train_paths): 7623
─── split para modelos ──────────────────────────────
  X_train_split              : 6097

In [3]:
# ============================================================
# Base comum para todos os experimentos (SEM DATA AUGMENTATION)
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================

import os
import numpy as np
import pandas as pd
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ============================================================
# 2. DOWNLOAD E LEITURA DO DATASET
# ============================================================

path = kagglehub.dataset_download("mahyeks/almond-varieties")
dataset_dir = os.path.join(path, "dataset")

valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

classes = sorted(
    nome for nome in os.listdir(dataset_dir)
    if os.path.isdir(os.path.join(dataset_dir, nome))
)

image_paths = []
labels = []

for classe in classes:
    classe_dir = os.path.join(dataset_dir, classe)

    for nome_arquivo in sorted(os.listdir(classe_dir)):
        if nome_arquivo.lower().endswith(valid_extensions):
            image_paths.append(os.path.join(classe_dir, nome_arquivo))
            labels.append(classe)

base_df = pd.DataFrame({
    "image_path": image_paths,
    "label": labels
})

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(base_df["label"])

print("Base carregada com sucesso.")
print(base_df["label"].value_counts())

# ============================================================
# 3. DIVISÃO TREINO / TESTE
# ============================================================

X_train_paths, X_test_paths, y_train, y_test = train_test_split(
    base_df["image_path"].values,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

print("\nDivisão treino/teste:")
print(f"Treino : {len(X_train_paths)} imagens")
print(f"Teste  : {len(X_test_paths)} imagens")

# ============================================================
# 4. DIVISÃO TREINO / VALIDAÇÃO
# ============================================================
# 60% treino
# 10% validação
# 30% teste
# ============================================================

X_train_split, X_val_paths, y_train_split, y_val = train_test_split(
    X_train_paths,
    y_train,
    test_size=(1/7),      # 10% da base total
    random_state=42,
    stratify=y_train
)

# ============================================================
# 5. RESUMO FINAL
# ============================================================

print("\n" + "=" * 55)
print("RESUMO DO DATASET")
print("=" * 55)

print(f"Treino     : {len(X_train_split)} imagens")
print(f"Validação  : {len(X_val_paths)} imagens")
print(f"Teste      : {len(X_test_paths)} imagens")

print("\nDistribuição por classe (treino):")

unique, counts = np.unique(y_train_split, return_counts=True)

for cls_idx, count in zip(unique, counts):
    print(f"{label_encoder.classes_[cls_idx]:<20}: {count}")

Using Colab cache for faster access to the 'almond-varieties' dataset.
Base carregada com sucesso.
label
KAPADOKYA    465
AK           401
SIRA         384
NURLU        306
Name: count, dtype: int64

Divisão treino/teste:
Treino : 1089 imagens
Teste  : 467 imagens

RESUMO DO DATASET
Treino     : 933 imagens
Validação  : 156 imagens
Teste      : 467 imagens

Distribuição por classe (treino):
AK                  : 241
KAPADOKYA           : 278
NURLU               : 183
SIRA                : 231


## DenseNet121

Baseline do artigo original. Use este bloco para a primeira comparação.

In [ ]:
# ============================================================
# DenseNet121 — Fine-tuning com carregamento lazy (tf.data)
#
# Pré-requisito: célula base_dataset_corrigido.py já executada.
# Variáveis consumidas da célula base:
#   X_train_split / y_train_split  → treino (orig + aumentadas)
#   X_val_paths   / y_val          → validação (só originais)
#   X_test_paths  / y_test         → teste (só originais)
#   label_encoder                  → encoder de classes
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import json
import pickle
import numpy as np
from datetime import datetime

import tensorflow as tf
from tensorflow.keras.applications.densenet import DenseNet121, preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import backend as K

from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, roc_auc_score, classification_report,
)

# ============================================================
# 2. LIBERAÇÃO DE MEMÓRIA
# ============================================================

try:
    del model_densenet
    K.clear_session()
    gc.collect()
    print("✓ Memória do modelo anterior liberada")
except Exception:
    pass

# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================

def load_and_preprocess_densenet(path, label):
    """
    Carrega e pré-processa UMA imagem com normalização DenseNet121.
    Chamada pelo tf.data somente quando o batch for necessário —
    nunca carrega o dataset inteiro na RAM.
    """
    img_raw = tf.io.read_file(path)
    img     = tf.image.decode_image(img_raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [224, 224])
    img = preprocess_input(tf.cast(img, tf.float32))
    return img, label


def criar_dataset_densenet(paths, labels, batch_size=32, shuffle=False):
    """
    Cria um tf.data.Dataset com carregamento lazy em disco.
    RAM utilizada por vez: batch_size × 224 × 224 × 3 × 4 bytes
    Ex.: 32 imagens ≈ 19 MB
    """
    dataset = tf.data.Dataset.from_tensor_slices(
        (list(map(str, paths)), labels.astype(np.int32))
    )
    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=2000, seed=42, reshuffle_each_iteration=True
        )
    return (
        dataset
        .map(load_and_preprocess_densenet, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )


def calcular_metricas_densenet(y_true, y_pred, y_pred_proba=None, num_classes=None):
    """Calcula métricas de classificação e retorna dicionário."""
    metricas = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="weighted",
                                     zero_division=0),
        "Recall":    recall_score(y_true, y_pred, average="weighted",
                                  zero_division=0),
        "F1-Score":  f1_score(y_true, y_pred, average="weighted",
                               zero_division=0),
        "Kappa":     cohen_kappa_score(y_true, y_pred),
    }
    if y_pred_proba is not None:
        if num_classes is None:
            num_classes = y_pred_proba.shape[1]
        if num_classes == 2:
            metricas["AUC"] = roc_auc_score(y_true, y_pred_proba[:, 1])
        else:
            y_true_bin = label_binarize(y_true, classes=range(num_classes))
            metricas["AUC"] = roc_auc_score(
                y_true_bin, y_pred_proba,
                multi_class="ovr", average="weighted",
            )
    else:
        metricas["AUC"] = None
    return metricas


def salvar_resultados_densenet(nome_modelo, metricas, y_true, y_pred,
                                y_pred_proba, history=None, diretorio="resultados"):
    """Salva métricas, predições e relatório em arquivos locais."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    pasta     = os.path.join(diretorio, f"{nome_modelo}_{timestamp}")
    os.makedirs(pasta, exist_ok=True)

    metricas_serial = {
        k: (float(v) if hasattr(v, "item") else v)
        for k, v in metricas.items()
    }

    with open(os.path.join(pasta, "metricas.json"), "w") as f:
        json.dump({"modelo": nome_modelo, "data_execucao": timestamp,
                   "metricas": metricas_serial}, f, indent=4)

    np.save(os.path.join(pasta, "y_true.npy"),  y_true)
    np.save(os.path.join(pasta, "y_pred.npy"),  y_pred)
    np.save(os.path.join(pasta, "y_proba.npy"), y_pred_proba)   # consistente com demais modelos

    with open(os.path.join(pasta, "classification_report.json"), "w") as f:
        json.dump(classification_report(y_true, y_pred, output_dict=True),
                  f, indent=4)

    if history is not None:
        hist_dict = {k: [float(v) for v in vals]
                     for k, vals in history.history.items()}
        with open(os.path.join(pasta, "training_history.json"), "w") as f:
            json.dump(hist_dict, f, indent=4)
        with open(os.path.join(pasta, "training_history.pkl"), "wb") as f:
            pickle.dump(history.history, f)

    with open(os.path.join(pasta, "resumo.txt"), "w", encoding="utf-8") as f:
        f.write("=" * 50 + "\n")
        f.write(f"RESUMO DO MODELO: {nome_modelo}\n")
        f.write("=" * 50 + "\n\n")
        f.write("MÉTRICAS:\n" + "-" * 30 + "\n")
        for metrica, valor in metricas_serial.items():
            f.write(f"{metrica}: {valor:.4f}\n" if valor is not None
                    else f"{metrica}: Não disponível\n")
        f.write("\nCONFIGURAÇÃO DO MODELO:\n" + "-" * 30 + "\n")
        f.write("Arquitetura: DenseNet121\n")
        f.write("Pesos: ImageNet (pré-treinado)\n")
        f.write("Carregamento: tf.data lazy (batch a batch)\n")
        f.write("Otimizador: Adam (lr=1e-4)\n")
        f.write("Loss: Sparse Categorical Crossentropy\n")
        f.write("Batch Size: 32\n")
        f.write("Epochs: até 50 (EarlyStopping patience=7)\n")
        f.write("Split: definido na célula base (StratifiedGroupKFold)\n")
        f.write("Validação: apenas imagens originais\n")

    print(f"\n✓ Resultados salvos em: {pasta}")
    return pasta


# ============================================================
# 4. DATASETS LAZY
# Split já realizado na célula base — variáveis prontas:
#   X_train_split / y_train_split → treino
#   X_val_paths   / y_val         → validação (só originais)
#   X_test_paths  / y_test        → teste     (só originais)
# ============================================================

BATCH_SIZE = 32

train_ds = criar_dataset_densenet(X_train_split, y_train_split,
                                   batch_size=BATCH_SIZE, shuffle=True)
val_ds   = criar_dataset_densenet(X_val_paths,   y_val,
                                   batch_size=BATCH_SIZE)
test_ds  = criar_dataset_densenet(X_test_paths,  y_test,
                                   batch_size=BATCH_SIZE)

ram_por_batch = BATCH_SIZE * 224 * 224 * 3 * 4 / 1e6
print(f"Treino    : {len(X_train_split):>6} imagens")
print(f"Validação : {len(X_val_paths):>6} imagens  (apenas originais)")
print(f"Teste     : {len(X_test_paths):>6} imagens  (apenas originais)")
print(f"\n✓ Datasets criados  |  RAM por batch: ~{ram_por_batch:.0f} MB")

# ============================================================
# 5. CONSTRUÇÃO DO MODELO
# ============================================================

print("\n" + "=" * 60)
print("CONSTRUÇÃO DO MODELO — DenseNet121")
print("=" * 60)

base_model = DenseNet121(weights="imagenet", include_top=False,
                          input_shape=(224, 224, 3))

for layer in base_model.layers:
    layer.trainable = False

x              = GlobalAveragePooling2D()(base_model.output)
x              = Dropout(0.3)(x)
output         = Dense(len(label_encoder.classes_), activation="softmax")(x)
model_densenet = Model(inputs=base_model.input, outputs=output)

model_densenet.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

treinavel = sum(p.numpy().size for p in model_densenet.trainable_weights)
total     = sum(p.numpy().size for p in model_densenet.weights)
print(f"✓ Parâmetros treináveis : {treinavel:,}  /  {total:,} total")

# ============================================================
# 6. TREINAMENTO
# ============================================================

os.makedirs("models", exist_ok=True)

callbacks = [
    EarlyStopping(
        monitor="val_loss", patience=7,
        restore_best_weights=True, verbose=1,
    ),
    ModelCheckpoint(
        "models/densenet121_best.keras", monitor="val_accuracy",
        save_best_only=True, verbose=0,
    ),
    ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3,
        min_lr=1e-7, verbose=1,
    ),
]

print("\n" + "=" * 60)
print("TREINAMENTO — DenseNet121")
print("=" * 60)

history = model_densenet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks,
    verbose=1,
)

model_densenet.save("models/densenet121.keras")
print("✓ Modelo salvo em: models/densenet121.keras")

# ============================================================
# 7. AVALIAÇÃO
# ============================================================

print("\n" + "=" * 60)
print("AVALIAÇÃO DO MODELO — DenseNet121")
print("=" * 60)

y_pred_proba_densenet = model_densenet.predict(test_ds, verbose=0)
y_pred_densenet       = np.argmax(y_pred_proba_densenet, axis=1)

metricas_densenet = calcular_metricas_densenet(
    y_test, y_pred_densenet, y_pred_proba_densenet,
    num_classes=len(label_encoder.classes_),
)

print("\n" + "=" * 50)
print("MÉTRICAS — DenseNet121")
print("=" * 50)
for metrica, valor in metricas_densenet.items():
    if valor is not None:
        print(f"  {metrica:<12}: {valor:.4f}")
    else:
        print(f"  {metrica:<12}: Não disponível")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_densenet,
                             target_names=label_encoder.classes_))

# ============================================================
# 8. SALVAR RESULTADOS
# ============================================================

pasta_resultados_densenet = salvar_resultados_densenet(
    nome_modelo="DenseNet121",
    metricas=metricas_densenet,
    y_true=y_test,
    y_pred=y_pred_densenet,
    y_pred_proba=y_pred_proba_densenet,
    history=history,
    diretorio="resultados",
)

# ============================================================
# 9. RESUMO FINAL
# ============================================================

print(f"\n{'=' * 60}")
print("RESUMO FINAL — DenseNet121")
print(f"{'=' * 60}")
print(f"✓ Acurácia : {metricas_densenet['Accuracy']:.4f}")
print(f"✓ Precision: {metricas_densenet['Precision']:.4f}")
print(f"✓ Recall   : {metricas_densenet['Recall']:.4f}")
print(f"✓ F1-Score : {metricas_densenet['F1-Score']:.4f}")
print(f"✓ Kappa    : {metricas_densenet['Kappa']:.4f}")
if metricas_densenet["AUC"] is not None:
    print(f"✓ AUC      : {metricas_densenet['AUC']:.4f}")
print(f"✓ Pasta    : {pasta_resultados_densenet}")
print(f"\nPróximo passo: Execute a célula de visualização.")

K.clear_session()
gc.collect()
print("\n✓ Sessão Keras limpa")

Treino    :    933 imagens
Validação :    156 imagens  (apenas originais)
Teste     :    467 imagens  (apenas originais)

✓ Datasets criados  |  RAM por batch: ~19 MB

CONSTRUÇÃO DO MODELO — DenseNet121
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
✓ Parâmetros treináveis : 4,100  /  7,041,604 total

TREINAMENTO — DenseNet121
Epoch 1/50
27/30 ━━━━━━━━━━━━━━━━━━━━ 21s 7s/step - accuracy: 0.2255 - loss: 1.9323

## ResNet50

CNN clássica e muito citada. Use este bloco para o segundo experimento.

In [ ]:
# ============================================================
# ResNet50 — Fine-tuning com carregamento lazy (tf.data)
#
# Pré-requisito: célula base_dataset_corrigido.py já executada.
# Variáveis consumidas da célula base:
#   X_train_split / y_train_split  → treino (orig + aumentadas)
#   X_val_paths   / y_val          → validação (só originais)
#   X_test_paths  / y_test         → teste (só originais)
#   label_encoder                  → encoder de classes
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import json
import pickle
import numpy as np
from datetime import datetime

import tensorflow as tf
from tensorflow.keras.applications.resnet50 import (
    ResNet50,
    preprocess_input as resnet_preprocess,
)
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import backend as K

from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, roc_auc_score, classification_report,
)

# ============================================================
# 2. LIBERAÇÃO DE MEMÓRIA
# ============================================================

try:
    del model
    K.clear_session()
    gc.collect()
    print("✓ Memória do modelo anterior liberada")
except Exception:
    pass

# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================

def load_and_preprocess_resnet(path, label):
    """
    Carrega e pré-processa UMA imagem com normalização ResNet50.
    Chamada pelo tf.data somente quando o batch for necessário —
    nunca carrega o dataset inteiro na RAM.
    """
    img_raw = tf.io.read_file(path)
    img     = tf.image.decode_image(img_raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [224, 224])
    img = resnet_preprocess(tf.cast(img, tf.float32))
    return img, label


def criar_dataset_resnet(paths, labels, batch_size=32, shuffle=False):
    """
    Cria um tf.data.Dataset com carregamento lazy em disco.
    RAM utilizada por vez: batch_size × 224 × 224 × 3 × 4 bytes
    Ex.: 32 imagens ≈ 19 MB
    """
    dataset = tf.data.Dataset.from_tensor_slices(
        (list(map(str, paths)), labels.astype(np.int32))
    )
    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=2000, seed=42, reshuffle_each_iteration=True
        )
    return (
        dataset
        .map(load_and_preprocess_resnet, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )


def calcular_metricas(y_true, y_pred, y_pred_proba=None, num_classes=None):
    """Calcula métricas de classificação e retorna dicionário."""
    metricas = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="weighted",
                                     zero_division=0),
        "Recall":    recall_score(y_true, y_pred, average="weighted",
                                  zero_division=0),
        "F1-Score":  f1_score(y_true, y_pred, average="weighted",
                               zero_division=0),
        "Kappa":     cohen_kappa_score(y_true, y_pred),
    }
    if y_pred_proba is not None:
        if num_classes is None:
            num_classes = y_pred_proba.shape[1]
        if num_classes == 2:
            metricas["AUC"] = roc_auc_score(y_true, y_pred_proba[:, 1])
        else:
            y_true_bin = label_binarize(y_true, classes=range(num_classes))
            metricas["AUC"] = roc_auc_score(
                y_true_bin, y_pred_proba,
                multi_class="ovr", average="weighted",
            )
    else:
        metricas["AUC"] = None
    return metricas


def salvar_resultados_resnet(nome_modelo, metricas, y_true, y_pred,
                              y_pred_proba, history=None, diretorio="resultados"):
    """Salva métricas, predições e relatório em arquivos locais."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    pasta     = os.path.join(diretorio, f"{nome_modelo}_{timestamp}")
    os.makedirs(pasta, exist_ok=True)

    metricas_serial = {
        k: (float(v) if hasattr(v, "item") else v)
        for k, v in metricas.items()
    }

    with open(os.path.join(pasta, "metricas.json"), "w") as f:
        json.dump({"modelo": nome_modelo, "data_execucao": timestamp,
                   "metricas": metricas_serial}, f, indent=4)

    np.save(os.path.join(pasta, "y_true.npy"),  y_true)
    np.save(os.path.join(pasta, "y_pred.npy"),  y_pred)
    np.save(os.path.join(pasta, "y_proba.npy"), y_pred_proba)

    with open(os.path.join(pasta, "classification_report.json"), "w") as f:
        json.dump(classification_report(y_true, y_pred, output_dict=True),
                  f, indent=4)

    if history is not None:
        hist_dict = {k: [float(v) for v in vals]
                     for k, vals in history.history.items()}
        with open(os.path.join(pasta, "training_history.json"), "w") as f:
            json.dump(hist_dict, f, indent=4)
        with open(os.path.join(pasta, "training_history.pkl"), "wb") as f:
            pickle.dump(history.history, f)

    with open(os.path.join(pasta, "resumo.txt"), "w", encoding="utf-8") as f:
        f.write("=" * 50 + "\n")
        f.write(f"RESUMO DO MODELO: {nome_modelo}\n")
        f.write("=" * 50 + "\n\n")
        f.write("MÉTRICAS:\n" + "-" * 30 + "\n")
        for metrica, valor in metricas_serial.items():
            f.write(f"{metrica}: {valor:.4f}\n" if valor is not None
                    else f"{metrica}: Não disponível\n")
        f.write("\nCONFIGURAÇÃO DO MODELO:\n" + "-" * 30 + "\n")
        f.write("Arquitetura: ResNet50\n")
        f.write("Pesos: ImageNet (pré-treinado)\n")
        f.write("Carregamento: tf.data lazy (batch a batch)\n")
        f.write("Otimizador: Adam (lr=1e-4)\n")
        f.write("Loss: Sparse Categorical Crossentropy\n")
        f.write("Batch Size: 32\n")
        f.write("Epochs: até 50 (EarlyStopping patience=7)\n")
        f.write("Split: definido na célula base (StratifiedGroupKFold)\n")
        f.write("Validação: apenas imagens originais\n")

    print(f"\n✓ Resultados salvos em: {pasta}")
    return pasta


# ============================================================
# 4. DATASETS LAZY
# Split já realizado na célula base — variáveis prontas:
#   X_train_split / y_train_split → treino
#   X_val_paths   / y_val         → validação (só originais)
#   X_test_paths  / y_test        → teste     (só originais)
# ============================================================

BATCH_SIZE = 32

train_ds = criar_dataset_resnet(X_train_split, y_train_split,
                                batch_size=BATCH_SIZE, shuffle=True)
val_ds   = criar_dataset_resnet(X_val_paths,   y_val,
                                batch_size=BATCH_SIZE)
test_ds  = criar_dataset_resnet(X_test_paths,  y_test,
                                batch_size=BATCH_SIZE)

ram_por_batch = BATCH_SIZE * 224 * 224 * 3 * 4 / 1e6
print(f"Treino    : {len(X_train_split):>6} imagens")
print(f"Validação : {len(X_val_paths):>6} imagens  (apenas originais)")
print(f"Teste     : {len(X_test_paths):>6} imagens  (apenas originais)")
print(f"\n✓ Datasets criados  |  RAM por batch: ~{ram_por_batch:.0f} MB")

# ============================================================
# 5. CONSTRUÇÃO DO MODELO
# ============================================================

print("\n" + "=" * 60)
print("CONSTRUÇÃO DO MODELO — ResNet50")
print("=" * 60)

base_model = ResNet50(weights="imagenet", include_top=False,
                      input_shape=(224, 224, 3))

for layer in base_model.layers:
    layer.trainable = False

x            = GlobalAveragePooling2D()(base_model.output)
x            = Dropout(0.3)(x)
output       = Dense(len(label_encoder.classes_), activation="softmax")(x)
model_resnet = Model(inputs=base_model.input, outputs=output)

model_resnet.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

total     = len(model_resnet.layers)
treinavel = len([l for l in model_resnet.layers if l.trainable])
print(f"✓ Total de camadas   : {total}")
print(f"✓ Camadas treináveis : {treinavel}")
print(f"✓ Camadas congeladas : {total - treinavel}")

# ============================================================
# 6. TREINAMENTO
# ============================================================

os.makedirs("models", exist_ok=True)

callbacks = [
    EarlyStopping(
        monitor="val_loss", patience=7,
        restore_best_weights=True, verbose=1,
    ),
    ModelCheckpoint(
        "models/resnet50_best.keras", monitor="val_accuracy",
        save_best_only=True, verbose=0,
    ),
    ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3,
        min_lr=1e-7, verbose=1,
    ),
]

print("\n" + "=" * 60)
print("TREINAMENTO — ResNet50")
print("=" * 60)

history = model_resnet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks,
    verbose=1,
)

model_resnet.save("models/resnet50.keras")
print("✓ Modelo salvo em: models/resnet50.keras")

# ============================================================
# 7. AVALIAÇÃO
# ============================================================

print("\n" + "=" * 60)
print("AVALIAÇÃO DO MODELO — ResNet50")
print("=" * 60)

y_pred_proba_resnet = model_resnet.predict(test_ds, verbose=0)
y_pred_resnet       = np.argmax(y_pred_proba_resnet, axis=1)

metricas_resnet = calcular_metricas(
    y_test, y_pred_resnet, y_pred_proba_resnet,
    num_classes=len(label_encoder.classes_),
)

print("\n" + "=" * 50)
print("MÉTRICAS — ResNet50")
print("=" * 50)
for metrica, valor in metricas_resnet.items():
    if valor is not None:
        print(f"  {metrica:<12}: {valor:.4f}")
    else:
        print(f"  {metrica:<12}: Não disponível")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_resnet,
                             target_names=label_encoder.classes_))

# ============================================================
# 8. SALVAR RESULTADOS
# ============================================================

pasta_resultados_resnet = salvar_resultados_resnet(
    nome_modelo="ResNet50",
    metricas=metricas_resnet,
    y_true=y_test,
    y_pred=y_pred_resnet,
    y_pred_proba=y_pred_proba_resnet,
    history=history,
    diretorio="resultados",
)

# ============================================================
# 9. RESUMO FINAL
# ============================================================

print(f"\n{'=' * 60}")
print("RESUMO FINAL — ResNet50")
print(f"{'=' * 60}")
print(f"✓ Acurácia : {metricas_resnet['Accuracy']:.4f}")
print(f"✓ Precision: {metricas_resnet['Precision']:.4f}")
print(f"✓ Recall   : {metricas_resnet['Recall']:.4f}")
print(f"✓ F1-Score : {metricas_resnet['F1-Score']:.4f}")
print(f"✓ Kappa    : {metricas_resnet['Kappa']:.4f}")
if metricas_resnet["AUC"] is not None:
    print(f"✓ AUC      : {metricas_resnet['AUC']:.4f}")
print(f"✓ Pasta    : {pasta_resultados_resnet}")
print(f"\nPróximo passo: Execute a célula de visualização.")

K.clear_session()
gc.collect()
print("\n✓ Sessão Keras limpa")

## ConvNeXt V2

CNN moderna. Use este bloco para o terceiro experimento.

In [ ]:
# ============================================================
# CONVNEXT V2
# ============================================================

import os
import copy
import time
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

from torchvision import transforms
from torchvision.models import convnext_base
from torchvision.models import ConvNeXt_Base_Weights

# ============================================================
# DISPOSITIVO
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


# ============================================================
# TRANSFORMAÇÕES
# ============================================================

weights = ConvNeXt_Base_Weights.DEFAULT

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights.meta["mean"] if "mean" in weights.meta else (0.485,0.456,0.406),
        std=weights.meta["std"] if "std" in weights.meta else (0.229,0.224,0.225)
    )
])

test_transform = train_transform

# ============================================================
# DATASET
# ============================================================

class AlmondDataset(Dataset):

    def __init__(self, image_paths, labels, transform=None):

        self.image_paths=image_paths
        self.labels=labels
        self.transform=transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image=Image.open(self.image_paths[idx]).convert("RGB")

        if self.transform:
            image=self.transform(image)

        label=self.labels[idx]

        return image,label

# ============================================================
# DATALOADERS
# ============================================================

batch_size=32

train_dataset=AlmondDataset(
    X_train_final,
    y_train_final,
    train_transform
)

val_dataset=AlmondDataset(
    X_val,
    y_val,
    test_transform
)

test_dataset=AlmondDataset(
    X_test_paths,
    y_test,
    test_transform
)

train_loader=DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

val_loader=DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

test_loader=DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

# ============================================================
# MODELO
# ============================================================

model=convnext_base(weights=weights)

num_features=model.classifier[2].in_features

model.classifier[2]=nn.Linear(
    num_features,
    len(label_encoder.classes_)
)

model=model.to(device)

# ============================================================
# OTIMIZADOR
# ============================================================

criterion=nn.CrossEntropyLoss()

optimizer=torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

# ============================================================
# TREINAMENTO
# ============================================================

epochs=20

best_model=copy.deepcopy(model.state_dict())
best_loss=np.inf

history_train=[]
history_val=[]

inicio=time.time()

for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    #########################
    # TREINO
    #########################

    model.train()

    train_loss=0

    for images,labels in train_loader:

        images=images.to(device)
        labels=labels.to(device)

        optimizer.zero_grad()

        outputs=model(images)

        loss=criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        train_loss+=loss.item()*images.size(0)

    train_loss/=len(train_loader.dataset)

    #########################
    # VALIDAÇÃO
    #########################

    model.eval()

    val_loss=0

    with torch.no_grad():

        for images,labels in val_loader:

            images=images.to(device)
            labels=labels.to(device)

            outputs=model(images)

            loss=criterion(outputs,labels)

            val_loss+=loss.item()*images.size(0)

    val_loss/=len(val_loader.dataset)

    scheduler.step(val_loss)

    history_train.append(train_loss)
    history_val.append(val_loss)

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_loss<best_loss:

        best_loss=val_loss
        best_model=copy.deepcopy(model.state_dict())

tempo=time.time()-inicio

print(f"\nTempo: {tempo/60:.2f} minutos")

# ============================================================
# MELHOR MODELO
# ============================================================

model.load_state_dict(best_model)

# ============================================================
# TESTE
# ============================================================

model.eval()

preds=[]
reais=[]

with torch.no_grad():

    for images,labels in test_loader:

        images=images.to(device)

        outputs=model(images)

        pred=torch.argmax(outputs,1)

        preds.extend(pred.cpu().numpy())
        reais.extend(labels.numpy())

# ============================================================
# MÉTRICAS
# ============================================================

acc=accuracy_score(reais,preds)

print("="*60)
print("RESULTADOS")
print("="*60)

print(f"Acurácia: {acc:.4f}")

print()

print(classification_report(
    reais,
    preds,
    target_names=label_encoder.classes_,
    digits=4
))

cm=confusion_matrix(reais,preds)

print("Matriz de Confusão")

print(cm)